
# Teste Llama + RAG (FAISS) para classificação em 5 classes

```


In [1]:

import os
import re
import json
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama

SEED = 8088135
random.seed(SEED)
np.random.seed(SEED)

# ─── Ficheiros ────────────────────────────────────────────────────────────────
EXAMPLE_CSV = 'dataset-exemplos.csv'   # opcional
PREDICT_CSV = 'subm2.csv'              # opcional
OUTPUT_CSV  = 'subm2_llama_rag.csv'
INDEX_DIR   = 'faiss_index_llama_rag'

# ─── Configuração ─────────────────────────────────────────────────────────────
PER_CLASS          = 4000   # baixa se quiseres indexar mais depressa
VAL_SIZE           = 0.10
EMBED_MODEL        = 'sentence-transformers/all-MiniLM-L6-v2'
OLLAMA_MODEL       = 'phi:latest'
TOP_K              = 5
TEMPERATURE        = 0.0
MAX_ROWS_TO_PREDICT = None   # None = prever tudo

CLASSES   = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
LABEL_NORM = {
    'anthropic': 'Anthropic', 'claude': 'Anthropic',
    'google': 'Google', 'gemini': 'Google', 'gemma': 'Google',
    'human': 'Human',
    'meta': 'Meta', 'llama': 'Meta',
    'openai': 'OpenAI', 'gpt': 'OpenAI', 'chatgpt': 'OpenAI',
}

print('Config carregada.')
print('Modelo Ollama:', OLLAMA_MODEL)
print('Embedding model:', EMBED_MODEL)
print('TOP_K:', TOP_K)


Config carregada.
Modelo Ollama: phi:latest
Embedding model: sentence-transformers/all-MiniLM-L6-v2
TOP_K: 5


In [2]:

# ─── Helpers ──────────────────────────────────────────────────────────────────
def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_label(label):
    label = str(label).strip().lower()
    return LABEL_NORM.get(label, label)

print('A carregar dataset do HuggingFace...')
df = load_dataset('BrunoFilipeRDS/50k-balanced-5-classes', split='train').to_pandas()
df.columns = df.columns.str.strip()
df = df.rename(columns={'text': 'Text', 'label': 'Labels'})

df['Text'] = df['Text'].astype(str).apply(clean_text)
df['Labels'] = df['Labels'].astype(str).map(normalize_label)
df = df[df['Labels'].isin(CLASSES)][['Text', 'Labels']].reset_index(drop=True)

print(f'Exemplos válidos: {len(df)}')
print(df['Labels'].value_counts().to_string())

frames = []
for label, group in df.groupby('Labels'):
    frames.append(group.sample(min(len(group), PER_CLASS), random_state=SEED))

df_small = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Após sample ({PER_CLASS}/classe): {len(df_small)} exemplos')
print(df_small['Labels'].value_counts().to_string())

X_train, X_val = train_test_split(
    df_small,
    test_size=VAL_SIZE,
    stratify=df_small['Labels'],
    random_state=SEED,
)
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)

print(f'Treino/indexação: {len(X_train)} | Validação: {len(X_val)}')
X_train.head()


A carregar dataset do HuggingFace...
Exemplos válidos: 250000
Labels
OpenAI       50000
Google       50000
Meta         50000
Anthropic    50000
Human        50000
Após sample (4000/classe): 20000 exemplos
Labels
Human        4000
OpenAI       4000
Google       4000
Anthropic    4000
Meta         4000
Treino/indexação: 18000 | Validação: 2000


,Text,Labels
0,Correct. Nitrogen exists as a diatomic molecul...,Anthropic
1,I will move my e2 pawn forward two squares to ...,Anthropic
2,Here's how to solve the problem step-by-step: ...,Google
3,Keshia Knight Pulliam announced her pregnancy ...,Google
4,"The new smartphone app, MindPal, offers a comp...",Meta


In [3]:

# ─── Construir Documents para o vectorstore ───────────────────────────────────
def row_to_document(row):
    content = (f"Label: {row['Labels']}"
        f"Text: {row['Text']}"
    )
    return Document(
        page_content=content,
        metadata={'label': row['Labels']}
    )

docs = [row_to_document(row) for _, row in X_train.iterrows()]
print('Documents criados:', len(docs))
print('Exemplo de documento:')
print(docs[0].page_content[:800])


Documents criados: 18000
Exemplo de documento:
Label: AnthropicText: Correct. Nitrogen exists as a diatomic molecule, meaning it consists of two atoms bonded together, with the chemical formula N2. Most elements exist in one of three main states at standard temperature and pressure: 1. Monoatomic - consisting of single atoms (e.g. noble gases like He, Ne, Ar, etc.) 2. Diatomic - consisting of pairs of atoms bonded together (e.g. N2, O2, H2, Cl2, etc.) 3. Polyatomic - consisting of more than two atoms (e.g. CO2, NO2, SO2, etc.) Gaseous nitrogen exists as N2: a diatomic molecule consisting of two nitrogen atoms bonded together. The chemical formula shows this as N2, with the subscript 2 indicating two nitrogen atoms.


In [4]:

# ─── Embeddings + FAISS ───────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

if os.path.isdir(INDEX_DIR):
    print(f'A carregar índice existente: {INDEX_DIR}')
    vectorstore = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
else:
    print('A criar índice FAISS...')
    vectorstore = FAISS.from_documents(docs, embeddings)
    vectorstore.save_local(INDEX_DIR)
    print(f'Índice guardado em: {INDEX_DIR}')

retriever = vectorstore.as_retriever(search_kwargs={'k': TOP_K})
print('Retriever pronto.')


C:\Users\bruno\AppData\Local\Temp\ipykernel_26852\773004069.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


A carregar índice existente: faiss_index_llama_rag
Retriever pronto.


In [5]:

# ─── Llama via Ollama ─────────────────────────────────────────────────────────
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=TEMPERATURE,
)

print('LLM pronto.')


LLM pronto.


In [6]:

# ─── Manual prompting with RAG ────────────────────────────────────────────────
def build_prompt(query_text, retrieved_docs):
    examples = []
    for i, doc in enumerate(retrieved_docs, start=1):
        examples.append(f"Example {i}: {doc.page_content}")
    examples_block = "\n\n".join(examples)

    prompt = f"""
You are a text classifier.

You must classify the user text into exactly ONE of these classes:
- Anthropic
- Google
- Human
- Meta
- OpenAI

Use the retrieved examples only as reference.

Similar examples:
{examples_block}

Text to classify:
Text: {query_text}

Mandatory rules:
1. Reply with only ONE label.
2. The reply must be exactly one of: Anthropic, Google, Human, Meta, OpenAI.
3. Do not explain.
4. Do not write full sentences.
""".strip()
    return prompt

def normalize_prediction(raw_text):
    txt = str(raw_text).strip()
    txt = txt.replace('*', '').replace('`', '').strip()

    # primeiro tenta match exato / contido nas labels
    for label in CLASSES:
        if txt == label:
            return label
    for label in CLASSES:
        if label.lower() in txt.lower():
            return label

    # fallback por normalização
    low = txt.lower()
    if 'anthropic' in low or 'claude' in low:
        return 'Anthropic'
    if 'google' in low or 'gemini' in low or 'gemma' in low:
        return 'Google'
    if 'human' in low or 'humano' in low:
        return 'Human'
    if 'meta' in low or 'llama' in low:
        return 'Meta'
    if 'openai' in low or 'gpt' in low or 'chatgpt' in low:
        return 'OpenAI'

    return 'Human'  # fallback conservador

def predict_label(query_text, return_debug=False):
    query_text = clean_text(query_text)
    retrieved_docs = retriever.invoke(query_text)
    prompt = build_prompt(query_text, retrieved_docs)
    response = llm.invoke(prompt)
    raw = response.content if hasattr(response, 'content') else str(response)
    pred = normalize_prediction(raw)

    if return_debug:
        return {
            'prediction': pred,
            'raw_response': raw,
            'retrieved_docs': retrieved_docs,
            'prompt': prompt,
        }
    return pred

print('Prediction function ready.')


Prediction function ready.


In [7]:

# ─── Teste rápido ─────────────────────────────────────────────────────────────
example_text = X_val.iloc[0]['Text']
result = predict_label(example_text, return_debug=True)

print('Predição:', result['prediction'])
print('Resposta bruta:', result['raw_response'])
print('Trecho do prompt:')
print(result['prompt'][:2000])


Predição: Meta
Resposta bruta:  MetaText:  ## Step 1: Recall the definition of a definite integral as the area under the curve of a function between two points. In this case, we have a polynomial function f(x) = x^4 + 2x^3 - 7x^2 + 4x + 5 and we want to find the definite integral from 

Trecho do prompt:
You are a text classifier.

You must classify the user text into exactly ONE of these classes:
- Anthropic
- Google
- Human
- Meta
- OpenAI

Use the retrieved examples only as reference.

Similar examples:
Example 1: Label: GoogleText: To find the antiderivative of f(x) = 5x^4 - 3x^2 + 2x, we apply the power rule for integration, which states that the integral of x^n is (x^(n+1))/(n+1) + C, where n ≠ -1 and C is the constant of integration. Applying this rule to each term in f(x): * **∫ 5x^4 dx:** (5x^(4+1))/(4+1) = (5x^5)/5 = x^5 * **∫ -3x^2 dx:** (-3x^(2+1))/(2+1) = (-3x^3)/3 = -x^3 * **∫ 2x dx:** (2x^(1+1))/(1+1) = (2x^2)/2 = x^2 Combining these results and adding the constant of in

In [8]:

# ─── Avaliação no set de validação ────────────────────────────────────────────
y_true = X_val['Labels'].tolist()
y_pred = []

for i, text in enumerate(X_val['Text'].tolist(), start=1):
    pred = predict_label(text)
    y_pred.append(pred)
    if i % 25 == 0 or i == len(X_val):
        print(f'{i}/{len(X_val)} processados')

acc = accuracy_score(y_true, y_pred)
print(f'Accuracy validação: {acc:.4f} ({acc*100:.2f}%)')
print('Classification report:')
print(classification_report(y_true, y_pred, labels=CLASSES, zero_division=0))

cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=CLASSES), index=CLASSES, columns=CLASSES)
cm


25/2000 processados
50/2000 processados
75/2000 processados
100/2000 processados
125/2000 processados
150/2000 processados
175/2000 processados
200/2000 processados
225/2000 processados
250/2000 processados
275/2000 processados
300/2000 processados
325/2000 processados
350/2000 processados
375/2000 processados
400/2000 processados
425/2000 processados
450/2000 processados
475/2000 processados
500/2000 processados
525/2000 processados
550/2000 processados
575/2000 processados
600/2000 processados
625/2000 processados
650/2000 processados
675/2000 processados
700/2000 processados
725/2000 processados
750/2000 processados
775/2000 processados
800/2000 processados
825/2000 processados
850/2000 processados
875/2000 processados
900/2000 processados
925/2000 processados
950/2000 processados
975/2000 processados
1000/2000 processados
1025/2000 processados
1050/2000 processados
1075/2000 processados
1100/2000 processados
1125/2000 processados
1150/2000 processados
1175/2000 processados
1200/200

,Anthropic,Google,Human,Meta,OpenAI
Anthropic,12,54,29,286,19
Google,13,88,27,256,16
Human,19,25,74,270,12
Meta,5,18,22,348,7
OpenAI,21,43,38,252,46



---
## (Opcional) Avaliar no dataset de exemplo do professor


In [10]:

try:
    df_ex = pd.read_csv(EXAMPLE_CSV, sep=';', encoding='utf-8-sig')
    df_ex.columns = df_ex.columns.str.strip()
    df_ex['Text'] = df_ex['Text'].astype(str).apply(clean_text)
    df_ex['Label'] = df_ex['Label'].astype(str).str.strip()
    df_ex = df_ex[df_ex['Label'].isin(CLASSES)].reset_index(drop=True)

    print(f'Exemplos do professor carregados: {len(df_ex)}')

    y_true_ex = df_ex['Label'].tolist()
    y_pred_ex = []
    for i, text in enumerate(df_ex['Text'].tolist(), start=1):
        y_pred_ex.append(predict_label(text))
        if i % 10 == 0 or i == len(df_ex):
            print(f'{i}/{len(df_ex)} processados')

    acc_ex = accuracy_score(y_true_ex, y_pred_ex)
    print(f'Accuracy: {acc_ex:.4f} ({acc_ex*100:.2f}%)')
    print(classification_report(y_true_ex, y_pred_ex, labels=CLASSES, zero_division=0))
except FileNotFoundError:
    print(f'Ficheiro {EXAMPLE_CSV} não encontrado — salta esta célula se não o tiveres.')


Exemplos do professor carregados: 125
10/125 processados
20/125 processados
30/125 processados
40/125 processados
50/125 processados
60/125 processados
70/125 processados
80/125 processados
90/125 processados
100/125 processados
110/125 processados
120/125 processados
125/125 processados
Accuracy: 0.1440 (14.40%)
              precision    recall  f1-score   support

   Anthropic       0.00      0.00      0.00        23
      Google       0.00      0.00      0.00        16
       Human       1.00      0.06      0.11        52
        Meta       0.14      0.76      0.24        17
      OpenAI       0.25      0.12      0.16        17

    accuracy                           0.14       125
   macro avg       0.28      0.19      0.10       125
weighted avg       0.47      0.14      0.10       125




---
## (Opcional) Prever o ficheiro de submissão


In [ ]:

try:
    df_pred = pd.read_csv(PREDICT_CSV, sep=';', encoding='utf-8-sig')
    df_pred.columns = df_pred.columns.str.strip()
    df_pred['Text'] = df_pred['Text'].astype(str).apply(clean_text)

    if MAX_ROWS_TO_PREDICT is not None:
        df_pred = df_pred.head(MAX_ROWS_TO_PREDICT).copy()

    print(f'Registos a prever: {len(df_pred)}')

    preds = []
    for i, text in enumerate(df_pred['Text'].tolist(), start=1):
        preds.append(predict_label(text))
        if i % 25 == 0 or i == len(df_pred):
            print(f'{i}/{len(df_pred)} processados')

    df_out = pd.DataFrame({
        'ID': df_pred['ID'],
        'Labels': preds,
    })
    df_out.to_csv(OUTPUT_CSV, index=False, sep=';')

    print(f'
Guardado: {OUTPUT_CSV}')
    print(df_out['Labels'].value_counts().to_string())
    display(df_out.head(10))
except FileNotFoundError:
    print(f'Ficheiro {PREDICT_CSV} não encontrado — salta esta célula se ainda não o tiveres.')



## Notas
- Esta abordagem é **mais lenta** do que o classificador DeBERTa porque faz retrieval + chamada ao Llama para cada texto.
- Se estiver demasiado lenta:
  - reduz `PER_CLASS`
  - reduz `TOP_K`
  - usa um modelo Ollama mais leve
  - testa apenas com parte do conjunto (`MAX_ROWS_TO_PREDICT`)
- Se o Ollama devolver texto extra, a função `normalize_prediction()` tenta limpar a resposta.
- Aqui o RAG está a ser usado como **few-shot dinâmico**: recupera exemplos semelhantes e injeta-os no prompt.
